# Evaluation: Fine-tuned Bank Transaction Model

Benchmarks the fine-tuned TinyLlama model (served via Ollama) across all task categories using the held-out test set (100 examples). Measures accuracy, response format adherence, and per-category quality.

## Step 1: Load Test Data & Setup

In [30]:
import sys
import json
import difflib
import requests
from pathlib import Path
from collections import defaultdict

RUNEBOOK_ROOT = Path.cwd()  # notebook 01's clone cell already chdir'd here
if str(RUNEBOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(RUNEBOOK_ROOT))

from src.data_loader import DataLoader

TEST_DATA = RUNEBOOK_ROOT / "data/curated/test.jsonl"
OLLAMA_MODEL = "tinyllama-finetuned:latest"
OLLAMA_URL = "http://localhost:11434/api/generate"

test_examples = DataLoader.load_dataset(TEST_DATA)
print(f"Loaded {len(test_examples)} test examples")
print(f"Model: {OLLAMA_MODEL}")

Loaded 100 test examples
Model: tinyllama-finetuned:latest


## Step 2: Run Inference on Full Test Set

In [ ]:
import difflib

def classify_task(instruction: str) -> str:
    i = instruction.lower()
    if "categorize" in i: return "Categorization"
    elif "fraud" in i: return "Fraud Detection"
    elif "spending insight" in i or "monthly transaction summary" in i: return "Spending Analysis"
    elif "customer question" in i: return "Customer Q&A"
    elif "pending transaction" in i: return "Tx Explanation"
    elif "dispute" in i or "disputing" in i: return "Dispute"
    elif "merchant" in i or "recognize" in i: return "Merchant ID"
    elif "savings" in i or "budget" in i or "financial" in i: return "Financial Plan"
    elif "declined" in i or "payment was" in i: return "Payment Failure"
    elif "security" in i: return "Security"
    else: return "Other"

def query_ollama(prompt: str, timeout: int = 5*60) -> str:
    resp = requests.post(OLLAMA_URL, json={"model": OLLAMA_MODEL, "prompt": prompt, "stream": False}, timeout=timeout)
    resp.raise_for_status()
    return resp.json()["response"]

results = []
for i, ex in enumerate(test_examples):
    prompt = f"{ex.instruction}\n{ex.input}"
    prediction = query_ollama(prompt)
    similarity = difflib.SequenceMatcher(None, prediction.strip().lower(), ex.output.strip().lower()).ratio()
    category = classify_task(ex.instruction)
    results.append({
        "category": category,
        "instruction": ex.instruction[:60],
        "expected": ex.output[:100],
        "predicted": prediction[:100],
        "similarity": similarity,
        "exact_match": prediction.strip() == ex.output.strip(),
    })
    if (i + 1) % 20 == 0:
        print(f"  Processed {i+1}/{len(test_examples)}...")

print(f"\nEvaluation complete: {len(results)} examples processed")

## Step 3: Overall Metrics

In [ ]:
import pandas as pd

avg_sim = sum(r["similarity"] for r in results) / len(results)
exact_matches = sum(r["exact_match"] for r in results)
high_quality = sum(1 for r in results if r["similarity"] > 0.6)

print(f"Overall Metrics ({len(results)} test examples):")
print(f"  Average similarity:     {avg_sim:.3f}")
print(f"  Exact match rate:       {exact_matches}/{len(results)} ({exact_matches/len(results)*100:.1f}%)")
print(f"  High quality (sim>0.6): {high_quality}/{len(results)} ({high_quality/len(results)*100:.1f}%)")
print(f"  Low quality (sim<0.3):  {sum(1 for r in results if r['similarity'] < 0.3)}/{len(results)}")

## Step 4: Per-Category Breakdown

In [ ]:
cat_metrics = defaultdict(list)
for r in results:
    cat_metrics[r["category"]].append(r["similarity"])

rows = []
for cat, sims in sorted(cat_metrics.items(), key=lambda x: -sum(x[1])/len(x[1])):
    rows.append({
        "Category": cat,
        "N": len(sims),
        "Avg Similarity": round(sum(sims) / len(sims), 3),
        "High Quality %": f"{sum(1 for s in sims if s > 0.6) / len(sims) * 100:.0f}%",
        "Min": round(min(sims), 3),
        "Max": round(max(sims), 3),
    })

df_cats = pd.DataFrame(rows)
display(df_cats)

## Step 5: Sample Predictions (Best & Worst)

In [ ]:
sorted_results = sorted(results, key=lambda r: r["similarity"], reverse=True)

print("=" * 70)
print("TOP 3 BEST PREDICTIONS")
print("=" * 70)
for r in sorted_results[:3]:
    print(f"\n[{r['category']}] Similarity: {r['similarity']:.3f}")
    print(f"  Expected:  {r['expected']}")
    print(f"  Predicted: {r['predicted']}")

print("\n" + "=" * 70)
print("TOP 3 WORST PREDICTIONS")
print("=" * 70)
for r in sorted_results[-3:]:
    print(f"\n[{r['category']}] Similarity: {r['similarity']:.3f}")
    print(f"  Expected:  {r['expected']}")
    print(f"  Predicted: {r['predicted']}")

## Step 5b: Out-of-Distribution Test (Unseen Prompts)

Tests the model on novel prompts NOT present in training data — different merchants, edge cases, and unusual scenarios to evaluate true generalization.

In [ ]:
# Prompts intentionally crafted to be OUTSIDE training distribution
unseen_prompts = [
    # 1. Unknown fintech merchant (not in training)
    {
        "instruction": "Categorize the following bank transaction based on its description.",
        "input": "NUBANK*PIX TRANSFER - $320,000 COP - 2024-08-22 16:45",
        "category": "Categorization",
        "notes": "Nubank not in training merchants; PIX is Brazilian payment system",
    },
    # 2. Crypto exchange (not in training categories)
    {
        "instruction": "Categorize the following bank transaction based on its description.",
        "input": "BINANCE.COM DEPOSIT - $2,500,000 COP - 2024-09-01 02:30",
        "category": "Categorization",
        "notes": "Crypto exchange not in merchant lists",
    },
    # 3. Fraud with novel pattern (layered attack)
    {
        "instruction": "Analyze this transaction and determine if it shows signs of potential fraud. Explain your reasoning.",
        "input": "Transaction: 3 purchases at APPLE STORE for exactly $999,000 each within 5 minutes, all using contactless tap. Customer profile: retired teacher, pension $2,800,000/month, never purchased Apple products before, average monthly spending $1,800,000 COP.",
        "category": "Fraud Detection",
        "notes": "Novel pattern: rapid contactless taps at same store (card cloning test)",
    },
    # 4. Customer Q&A about a product NOT in training
    {
        "instruction": "Answer the following customer question about their bank transactions.",
        "input": "I see a charge of $4,900 from 'CHATGPT PLUS' but I use the free version. Is this a scam?",
        "category": "Customer Q&A",
        "notes": "ChatGPT subscription not in training data",
    },
    # 5. Merchant ID with obfuscated name
    {
        "instruction": "The customer doesn't recognize this merchant name on their statement. Help identify it.",
        "input": "Unknown charge: 'DLOCAL*SHEIN' - $89,000 COP - 2024-07-10 14:20",
        "category": "Merchant ID",
        "notes": "Shein via dLocal not in training mappings",
    },
    # 6. Dispute with complex multi-party scenario
    {
        "instruction": "Guide the customer through disputing the following transaction.",
        "input": "Customer says: I ordered food via Rappi but the restaurant sent the wrong order. Rappi says contact the restaurant, the restaurant says contact Rappi. I was charged $65,000 and nobody will refund me.",
        "category": "Dispute",
        "notes": "Three-party dispute scenario not templated in training",
    },
    # 7. Payment failure with unusual reason
    {
        "instruction": "Explain why this payment was declined and suggest next steps.",
        "input": "Declined: AVIANCA.COM - $1,850,000 COP - Reason: 3DS_AUTHENTICATION_FAILED. Balance: $5,000,000 (sufficient).",
        "category": "Payment Failure",
        "notes": "3DS auth failure code not in training decline reasons",
    },
    # 8. Financial planning with unusual goal
    {
        "instruction": "Based on the customer's transaction history, suggest a personalized savings plan.",
        "input": "Profile: Income $12,000,000/month, expenses $11,200,000. Current savings: $45,000,000. Goal: early retirement in 3 years, needs $500,000,000 in investments. Spends $2,000,000/month on luxury dining and $1,500,000 on international travel.",
        "category": "Financial Plan",
        "notes": "High-income early retirement scenario outside normal profiles",
    },
    # 9. Security event with social engineering
    {
        "instruction": "Review these recent account security events and advise the customer on their account safety.",
        "input": "Security log: Customer called support 3 times today claiming to be locked out. Each call from a different phone number. Customer's actual device shows no login attempts. A $5,000,000 transfer was requested via phone banking during the 3rd call.",
        "category": "Security",
        "notes": "Voice phishing / social engineering via call center not in training",
    },
    # 10. Transaction explanation with novel status
    {
        "instruction": "Explain what the following pending transaction means and when it will be finalized.",
        "input": "REVERSED: UBER *TRIP - $28,000 COP - Status: Refund after fare adjustment. Original charge was $45,000.",
        "category": "Tx Explanation",
        "notes": "Partial reversal/fare adjustment not in training scenarios",
    },
]

print(f"Testing {len(unseen_prompts)} out-of-distribution prompts...\n")
print("=" * 70)

ood_results = []
for i, item in enumerate(unseen_prompts):
    prompt = f"{item['instruction']}\n{item['input']}"
    response = query_ollama(prompt)
    ood_results.append({"category": item["category"], "notes": item["notes"], "response": response})

    print(f"\n[{i+1}] {item['category']} -- {item['notes']}")
    print(f"    Input:    {item['input'][:90]}...")
    print(f"    Response: {response[:200]}")
    print("-" * 70)

print("\nOut-of-distribution test complete")

## Step 6: Similarity Distribution

In [0]:
import matplotlib.pyplot as plt

sims = [r["similarity"] for r in results]

plt.figure(figsize=(8, 4))
plt.hist(sims, bins=20, color="seagreen", edgecolor="white", alpha=0.8)
plt.axvline(x=0.6, color="red", linestyle="--", label="High quality threshold (0.6)")
plt.xlabel("Similarity Score")
plt.ylabel("Count")
plt.title("Distribution of Response Similarity (Test Set)")
plt.legend()
plt.tight_layout()
plt.show()

print(f"\nMedian similarity: {sorted(sims)[len(sims)//2]:.3f}")
print(f"Mean similarity:   {sum(sims)/len(sims):.3f}")

## Evaluation Complete

The fine-tuned TinyLlama model has been benchmarked across all task categories on the held-out test set. Key metrics: average similarity, per-category quality, and best/worst prediction samples provide a full picture of model performance.